<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Autoenk%C3%B3derek%20%C3%A9s%20Sparse%20Autoenk%C3%B3derek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autoenkóderek és Sparse Autoenkóderek

Ebben a notebookban az **autoencoder** architektúrákat vizsgáljuk.

## Tartalomjegyzék

1. Autoencoder alapok
2. Latent Space
3. Sparse Autoencoders
4. Denoising Autoencoders
5. Variational Autoencoders (VAE)

## 1. Autoencoder alapok

### Architektúra

```
Input → [Encoder] → Latent (bottleneck) → [Decoder] → Reconstruction
  x         f(x)          z                  g(z)          x̂
```

### Veszteségfüggvény

$$\mathcal{L}_{recon} = \|x - \hat{x}\|^2 = \|x - g(f(x))\|^2$$

### Cél

| Alkalmazás | Leírás |
|-----------|--------|
| Dimenzió csökkentés | PCA nemlineáris verziója |
| Feature learning | Reprezentáció tanulás |
| Anomaly detection | Rekonstrukciós hiba alapján |
| Generatív modell | Latent space-ből mintavétel |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

# Egyszerű Autoencoder

class SimpleAutoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, latent_dim=32):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU()
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()  # [0, 1] output
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

# Szintetikus adat (MNIST-szerű)
def generate_digits(n_samples=1000):
    """Egyszerű digit-szerű képek generálása."""
    images = []
    labels = []

    for i in range(n_samples):
        img = np.zeros((28, 28))
        digit = i % 3  # 0, 1, 2

        if digit == 0:  # Kör
            cx, cy = 14 + np.random.randint(-3, 4), 14 + np.random.randint(-3, 4)
            r = 6 + np.random.randint(-2, 3)
            for x in range(28):
                for y in range(28):
                    if abs((x-cx)**2 + (y-cy)**2 - r**2) < 20:
                        img[x, y] = 1
        elif digit == 1:  # Függőleges vonal
            col = 14 + np.random.randint(-4, 5)
            img[5:23, col-1:col+2] = 1
        else:  # Kereszt
            cx, cy = 14 + np.random.randint(-3, 4), 14 + np.random.randint(-3, 4)
            img[cx-5:cx+6, cy-1:cy+2] = 1
            img[cx-1:cx+2, cy-5:cy+6] = 1

        # Zaj hozzáadása
        img += np.random.randn(28, 28) * 0.1
        img = np.clip(img, 0, 1)

        images.append(img)
        labels.append(digit)

    return np.array(images), np.array(labels)

X, y = generate_digits(600)
X_tensor = torch.FloatTensor(X.reshape(-1, 784))
y_tensor = torch.LongTensor(y)

# Vizualizáció
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
for i, digit in enumerate([0, 1, 2]):
    idx = np.where(y == digit)[0][:5]
    for j, k in enumerate(idx):
        axes[i, j].imshow(X[k], cmap='gray')
        axes[i, j].axis('off')
    axes[i, 0].set_ylabel(f'Class {digit}', fontsize=12)
plt.suptitle('Szintetikus adat', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Autoencoder tanítás

def train_autoencoder(model, X, epochs=100, lr=0.001, batch_size=32):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    dataset = TensorDataset(X)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    losses = []

    for epoch in range(epochs):
        epoch_loss = 0
        for (batch,) in loader:
            optimizer.zero_grad()
            x_recon, _ = model(batch)
            loss = criterion(x_recon, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        losses.append(epoch_loss / len(loader))

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}: Loss = {losses[-1]:.4f}")

    return losses

# Tanítás
autoencoder = SimpleAutoencoder(input_dim=784, hidden_dim=128, latent_dim=16)
losses = train_autoencoder(autoencoder, X_tensor, epochs=100)

plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss')
plt.title('Autoencoder tanulási görbe')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Rekonstrukció vizualizáció

autoencoder.eval()
with torch.no_grad():
    x_recon, z = autoencoder(X_tensor[:9])

fig, axes = plt.subplots(2, 9, figsize=(18, 4))

for i in range(9):
    # Eredeti
    axes[0, i].imshow(X[i], cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Eredeti')

    # Rekonstrukció
    axes[1, i].imshow(x_recon[i].reshape(28, 28).numpy(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Rekonstrukció')

plt.suptitle('Autoencoder rekonstrukció', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Latent Space

### Tulajdonságok

- Tömörített reprezentáció
- Hasonló inputok közel a latent space-ben
- Dimenzió csökkentés

In [ ]:
# Latent space vizualizáció

autoencoder.eval()
with torch.no_grad():
    _, Z = autoencoder(X_tensor)
    Z = Z.numpy()

# PCA for 2D visualization
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
Z_2d = pca.fit_transform(Z)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=y, cmap='Set1', alpha=0.7, s=30)
plt.colorbar(scatter, label='Class')
plt.xlabel('Latent dim 1 (PCA)')
plt.ylabel('Latent dim 2 (PCA)')
plt.title('Latent Space vizualizáció')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Latent space interpoláció

def interpolate_latent(model, z1, z2, n_steps=10):
    """Két pont közötti interpoláció a latent space-ben."""
    alphas = np.linspace(0, 1, n_steps)
    interpolations = []

    for alpha in alphas:
        z_interp = (1 - alpha) * z1 + alpha * z2
        with torch.no_grad():
            x_interp = model.decode(z_interp)
        interpolations.append(x_interp.numpy().reshape(28, 28))

    return interpolations

# Két különböző osztályból
idx_0 = np.where(y == 0)[0][0]  # Kör
idx_1 = np.where(y == 1)[0][0]  # Vonal

with torch.no_grad():
    z1 = autoencoder.encode(X_tensor[idx_0:idx_0+1])
    z2 = autoencoder.encode(X_tensor[idx_1:idx_1+1])

interpolations = interpolate_latent(autoencoder, z1, z2, n_steps=10)

fig, axes = plt.subplots(1, 10, figsize=(20, 2))
for i, img in enumerate(interpolations):
    axes[i].imshow(img, cmap='gray')
    axes[i].axis('off')
    if i == 0:
        axes[i].set_title('Start')
    elif i == 9:
        axes[i].set_title('End')

plt.suptitle('Latent Space interpoláció', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Sparse Autoencoders

### Motiváció

Kikényszerítjük, hogy a latent kód sparse legyen (sok 0, kevés aktív neuron).

### Sparsity constraint

**L1 regularizáció:**
$$\mathcal{L} = \|x - \hat{x}\|^2 + \lambda \|z\|_1$$

**KL divergencia:**
$$\mathcal{L} = \|x - \hat{x}\|^2 + \beta \sum_j KL(\rho \| \hat{\rho}_j)$$

ahol $\rho$ a cél aktiváció (pl. 0.05), $\hat{\rho}_j$ az átlagos aktiváció.

In [ ]:
# Sparse Autoencoder L1-gyel

class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, latent_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.Sigmoid()  # [0, 1] aktivációk
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

def train_sparse_autoencoder(model, X, sparsity_weight=0.01, epochs=100, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    dataset = TensorDataset(X)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    losses = []
    sparsities = []

    for epoch in range(epochs):
        epoch_loss = 0
        epoch_sparsity = 0

        for (batch,) in loader:
            optimizer.zero_grad()
            x_recon, z = model(batch)

            # Reconstruction loss
            recon_loss = F.mse_loss(x_recon, batch)

            # L1 sparsity loss
            sparsity_loss = torch.mean(torch.abs(z))

            # Total loss
            loss = recon_loss + sparsity_weight * sparsity_loss

            loss.backward()
            optimizer.step()

            epoch_loss += recon_loss.item()
            epoch_sparsity += (z > 0.1).float().mean().item()

        losses.append(epoch_loss / len(loader))
        sparsities.append(epoch_sparsity / len(loader))

    return losses, sparsities

# Összehasonlítás különböző sparsity weight-ekkel
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, sw in enumerate([0, 0.01, 0.1]):
    torch.manual_seed(42)
    sparse_ae = SparseAutoencoder(input_dim=784, latent_dim=64)
    losses, sparsities = train_sparse_autoencoder(sparse_ae, X_tensor, sparsity_weight=sw, epochs=100)

    # Latent aktivációk
    sparse_ae.eval()
    with torch.no_grad():
        _, z = sparse_ae(X_tensor[:100])

    axes[i].hist(z.numpy().flatten(), bins=50, density=True, alpha=0.7)
    axes[i].set_xlabel('Latent aktiváció')
    axes[i].set_ylabel('Density')
    axes[i].set_title(f'Sparsity weight = {sw}\nAktív (>0.1): {(z > 0.1).float().mean():.1%}')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Sparse Autoencoder L1 regularizációval', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# KL divergencia sparse autoencoder

def kl_divergence(rho, rho_hat):
    """KL divergencia a cél és tényleges aktiváció között."""
    return rho * torch.log(rho / rho_hat) + (1 - rho) * torch.log((1 - rho) / (1 - rho_hat))

class KLSparseAutoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, latent_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.Sigmoid()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

def train_kl_sparse(model, X, rho=0.05, beta=3, epochs=100):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    dataset = TensorDataset(X)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    losses = []

    for epoch in range(epochs):
        epoch_loss = 0

        for (batch,) in loader:
            optimizer.zero_grad()
            x_recon, z = model(batch)

            # Reconstruction loss
            recon_loss = F.mse_loss(x_recon, batch)

            # KL divergence sparsity
            rho_hat = z.mean(dim=0)  # Átlagos aktiváció feature-önként
            rho_hat = torch.clamp(rho_hat, 1e-6, 1-1e-6)  # Numerikus stabilitás
            kl_loss = kl_divergence(torch.tensor(rho), rho_hat).sum()

            loss = recon_loss + beta * kl_loss
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        losses.append(epoch_loss / len(loader))

    return losses

# Tanítás
torch.manual_seed(42)
kl_sparse_ae = KLSparseAutoencoder(input_dim=784, latent_dim=64)
losses = train_kl_sparse(kl_sparse_ae, X_tensor, rho=0.05, beta=3, epochs=100)

# Aktivációk eloszlása
kl_sparse_ae.eval()
with torch.no_grad():
    _, z = kl_sparse_ae(X_tensor)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses, linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('KL Sparse Autoencoder tanulás')
axes[0].grid(True, alpha=0.3)

axes[1].hist(z.numpy().flatten(), bins=50, density=True, alpha=0.7)
axes[1].axvline(x=0.05, color='r', linestyle='--', label=f'Target ρ = 0.05')
axes[1].set_xlabel('Latent aktiváció')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Aktiváció eloszlás\nMean: {z.mean():.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Denoising Autoencoders

### Ötlet

Input: zajos kép
Target: tiszta kép

$$\mathcal{L} = \|x - g(f(\tilde{x}))\|^2$$

ahol $\tilde{x} = x + \epsilon$, $\epsilon \sim \mathcal{N}(0, \sigma^2)$

### Előnyök

- Robusztusabb reprezentáció
- Jobb generalizáció
- Implicit regularizáció

In [ ]:
# Denoising Autoencoder

class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

def add_noise(x, noise_factor=0.3):
    """Gaussian zaj hozzáadása."""
    noise = torch.randn_like(x) * noise_factor
    return torch.clamp(x + noise, 0, 1)

def train_denoising_ae(model, X, noise_factor=0.3, epochs=100):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    dataset = TensorDataset(X)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    losses = []

    for epoch in range(epochs):
        epoch_loss = 0

        for (batch,) in loader:
            optimizer.zero_grad()

            # Zaj hozzáadása az inputhoz
            noisy_batch = add_noise(batch, noise_factor)

            # Rekonstrukció a zajos inputból
            x_recon, _ = model(noisy_batch)

            # Loss: rekonstruáld a TISZTA képet
            loss = F.mse_loss(x_recon, batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        losses.append(epoch_loss / len(loader))

    return losses

# Tanítás
torch.manual_seed(42)
dae = DenoisingAutoencoder()
losses = train_denoising_ae(dae, X_tensor, noise_factor=0.4, epochs=150)

plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Denoising Autoencoder tanulás')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Denoising eredmények

dae.eval()

# Test képek
test_idx = [0, 1, 2, 3, 4]
X_test = X_tensor[test_idx]
X_noisy = add_noise(X_test, noise_factor=0.4)

with torch.no_grad():
    X_denoised, _ = dae(X_noisy)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for i in range(5):
    # Eredeti
    axes[0, i].imshow(X_test[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    if i == 2:
        axes[0, i].set_title('Eredeti', fontsize=12)

    # Zajos
    axes[1, i].imshow(X_noisy[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    if i == 2:
        axes[1, i].set_title('Zajos', fontsize=12)

    # Denoised
    axes[2, i].imshow(X_denoised[i].reshape(28, 28), cmap='gray')
    axes[2, i].axis('off')
    if i == 2:
        axes[2, i].set_title('Denoised', fontsize=12)

plt.suptitle('Denoising Autoencoder eredmény', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Variational Autoencoders (VAE)

### Különbség a sima AE-től

| Autoencoder | VAE |
|------------|-----|
| Determinisztikus z | Valószínűségi z |
| Pontszerű latent | Eloszlás a latent-en |
| Csak rekonstrukció | Rekonstrukció + KL |

### VAE loss

$$\mathcal{L}_{VAE} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) \| p(z))$$

**Reconstruction term:**
$$\mathbb{E}_{q(z|x)}[\log p(x|z)] \approx -\|x - \hat{x}\|^2$$

**KL term (analitikus, ha Gaussian):**
$$D_{KL}(q(z|x) \| p(z)) = -\frac{1}{2}\sum_{j=1}^{J}(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2)$$

### Reparametrization trick

$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

In [ ]:
# Variational Autoencoder

class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=16):
        super().__init__()

        # Encoder -> mu, log_var
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        log_var = self.fc_logvar(h)
        return mu, log_var

    def reparameterize(self, mu, log_var):
        """Reparametrization trick."""
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_recon = self.decode(z)
        return x_recon, mu, log_var

def vae_loss(x_recon, x, mu, log_var):
    """VAE loss: reconstruction + KL divergence."""
    # Reconstruction loss
    recon_loss = F.binary_cross_entropy(x_recon, x, reduction='sum')

    # KL divergence
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())

    return recon_loss + kl_loss

def train_vae(model, X, epochs=150):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    dataset = TensorDataset(X)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    losses = []

    for epoch in range(epochs):
        epoch_loss = 0

        for (batch,) in loader:
            optimizer.zero_grad()
            x_recon, mu, log_var = model(batch)
            loss = vae_loss(x_recon, batch, mu, log_var)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        losses.append(epoch_loss / len(X))

        if (epoch + 1) % 30 == 0:
            print(f"Epoch {epoch+1}: Loss = {losses[-1]:.2f}")

    return losses

# Tanítás
torch.manual_seed(42)
vae = VAE(input_dim=784, latent_dim=8)
losses = train_vae(vae, X_tensor, epochs=150)

In [ ]:
# VAE generálás

vae.eval()

# Random minták a latent space-ből
with torch.no_grad():
    z_samples = torch.randn(16, 8)  # Standard normal
    generated = vae.decode(z_samples)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))

for i, ax in enumerate(axes.ravel()):
    ax.imshow(generated[i].reshape(28, 28).numpy(), cmap='gray')
    ax.axis('off')

plt.suptitle('VAE generált képek (random sampling)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# VAE latent space (2D)

# Újratanítás 2D latent-tel
torch.manual_seed(42)
vae_2d = VAE(input_dim=784, latent_dim=2)
_ = train_vae(vae_2d, X_tensor, epochs=200)

# Latent space vizualizáció
vae_2d.eval()
with torch.no_grad():
    mu, _ = vae_2d.encode(X_tensor)
    mu = mu.numpy()

plt.figure(figsize=(10, 8))
scatter = plt.scatter(mu[:, 0], mu[:, 1], c=y, cmap='Set1', alpha=0.7, s=40)
plt.colorbar(scatter, label='Class')
plt.xlabel('μ₁')
plt.ylabel('μ₂')
plt.title('VAE 2D Latent Space')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 2D latent space grid generálás

n_grid = 15
z1 = np.linspace(-3, 3, n_grid)
z2 = np.linspace(-3, 3, n_grid)

fig, axes = plt.subplots(n_grid, n_grid, figsize=(15, 15))

for i, z1_val in enumerate(z1):
    for j, z2_val in enumerate(z2):
        z = torch.FloatTensor([[z1_val, z2_val]])
        with torch.no_grad():
            img = vae_2d.decode(z)

        axes[n_grid-1-j, i].imshow(img.reshape(28, 28).numpy(), cmap='gray')
        axes[n_grid-1-j, i].axis('off')

plt.suptitle('VAE 2D Latent Space Grid', fontsize=16)
plt.tight_layout()
plt.show()

## Összefoglalás

### Autoencoder típusok

| Típus | Loss | Cél |
|-------|------|-----|
| Simple AE | MSE | Rekonstrukció |
| Sparse AE | MSE + L1/KL | Sparse kód |
| Denoising AE | MSE (zajos → tiszta) | Robusztus feature |
| VAE | ELBO (recon + KL) | Generatív modell |

### VAE vs AE

| Tulajdonság | AE | VAE |
|------------|-------|-----|
| Latent | Determinisztikus | Valószínűségi |
| Generálás | Nehéz | Könnyű (prior sampling) |
| Loss | MSE | ELBO |
| Latent space | Nem szabályos | Regularizált |

### PyTorch implementáció

```python
# Reparametrization trick
z = mu + torch.exp(0.5 * log_var) * torch.randn_like(log_var)

# VAE loss
recon = F.binary_cross_entropy(x_recon, x)
kl = -0.5 * (1 + log_var - mu**2 - log_var.exp()).sum()
```